# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset and metadata
dataset = mlc.Dataset(url)
# Metadata is an mlcroissant object (not dict), print out common fields
print(f"{dataset.metadata.name}: {dataset.metadata.description}\n")
print(f"Published: {getattr(dataset.metadata, 'date_published', None)}\n")
print(f"Cite as: {getattr(dataset.metadata, 'cite_as', None)}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their fields
record_sets = dataset.metadata.record_sets
print(f"Available record sets: {len(record_sets)}\n")
for idx, rs in enumerate(record_sets):
    print(f"[{idx}] RecordSet name: {rs.name}, @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        field_label = field.label if hasattr(field, 'label') else field.name
        print(f"    - {field_label or field.name} (@id: {field.id})  | dataType: {getattr(field, 'data_type', None)}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use record set and field `@id`s as referenced above.

In [ ]:
# Build a list of all RecordSet @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
# Load records from each record set to a DataFrame
dataframes = {}
for record_set_id in record_set_ids:
    records_iter = dataset.records(record_set=record_set_id)
    records = list(records_iter)
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for RecordSet @id: {record_set_id}")

# Print column names of the first record set as an example
if record_set_ids:
    print("\nColumns/fields in first RecordSet:")
    print(dataframes[record_set_ids[0]].columns.tolist())
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes filtering, normalizing, and grouping operations.

Replace `<numeric_field_id>` and `<group_field_id>` with actual `@id`s as available in your record set.

In [ ]:
# Example: Pick a RecordSet and fields for analysis
chosen_record_set_id = record_set_ids[0] if record_set_ids else None
# Select a numeric field: find one with data_type float or int
record_set = next((rs for rs in dataset.metadata.record_sets if rs.id == chosen_record_set_id), None)

numeric_field_id = None
for field in record_set.fields:
    if getattr(field, 'data_type', None) in ('Float', 'Integer', 'Number'):
        numeric_field_id = field.id
        break
if numeric_field_id is None:
    print("No numeric field found.")
else:
    print(f"Analyzing numeric field: {numeric_field_id}")
    df = dataframes[chosen_record_set_id].copy()
    # Filter out missing values and parse column as numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold} (mean): {len(filtered_df)} rows\n")
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to group by a non-numeric field—find one with Text type
    group_field_id = None
    for field in record_set.fields:
        if getattr(field, 'data_type', None) == 'Text':
            group_field_id = field.id
            break

    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped average of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. The plot below shows a histogram for the chosen numeric field (if present).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_record_set_id and numeric_field_id and numeric_field_id in dataframes[chosen_record_set_id]:
    df = dataframes[chosen_record_set_id]
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
else:
    print('No suitable numeric field for visualization.')

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 protein abundance dataset using `mlcroissant`. You have seen how to extract record sets, identify schema by `@id`, examine field types, filter numeric columns, normalize, group, and visualize data. These steps form a robust workflow for working with Croissant-compatible datasets for further machine learning or data analysis work.